# Canadian PII NER — end-to-end (generate data → train → evaluate)

Runs the whole pipeline from scratch: upload your generator → produce JSONL →
convert to spaCy `.spacy` → frozen warm-up → unfrozen fine-tune → evaluate →
smoke test → download.

Optimizer blocks are real `[training.optimizer]` (LR actually applies), stage 2
sources both components from the frozen model, and step caps are safety ceilings
with patience doing the early stopping. Mount Drive so a disconnect doesn't wipe
your work.

## (Optional) Mount Drive so outputs persist

In [ ]:
from google.colab import drive; drive.mount('/content/drive')
import os; os.makedirs('/content/drive/MyDrive/pii_ner', exist_ok=True)
%cd /content/drive/MyDrive/pii_ner
print('cwd:', os.getcwd())

## 1. Install

In [ ]:
!pip install -q spacy==3.7.5
!python -m spacy download en_core_web_lg

## 2. Upload your generator (`synthetic_ner_generator.py`)

In [ ]:
from google.colab import files
up = files.upload()          # choose synthetic_ner_generator.py
assert 'synthetic_ner_generator.py' in up, 'upload synthetic_ner_generator.py'

## 3. Generate the synthetic corpus (JSONL + label.json)
Adjust sizes as you like; defaults come from the script.

In [ ]:
!python synthetic_ner_generator.py --output-dir synth --seed 42 \
    --train-size 12000 --validation-size 1600 --test-size 1600
!ls -la synth

## 4. Convert JSONL → spaCy `.spacy` (BIO tags → entity spans)

In [ ]:
import json, os, spacy
from spacy.tokens import Doc, DocBin, Span

with open("synth/label.json") as f:
    label2id = json.load(f)
id2label = {v: k for k, v in label2id.items()}
vocab = spacy.blank("en").vocab

def convert(src, dst):
    db = DocBin(); n = 0
    with open(src) as f:
        for line in f:
            ex = json.loads(line)
            words = ex["tokens"]
            tags = [id2label[int(t)] for t in ex["tags"]]
            doc = Doc(vocab, words=words)
            spans, i = [], 0
            while i < len(tags):
                t = tags[i]
                if t.startswith("B-"):
                    lab = t[2:]; j = i + 1
                    while j < len(tags) and tags[j] == f"I-{lab}": j += 1
                    spans.append(Span(doc, i, j, label=lab)); i = j
                else:
                    i += 1
            doc.ents = spans
            db.add(doc); n += 1
    db.to_disk(dst); print(f"{src} -> {dst}  ({n} docs)")

os.makedirs("corpus", exist_ok=True)
convert("synth/train.jsonl",      "corpus/train.spacy")
convert("synth/validation.jsonl", "corpus/valid.spacy")
convert("synth/test.jsonl",       "corpus/test.spacy")

## 5. Stage-1 config (frozen) + train

In [ ]:
%%writefile base_config.cfg
[paths]
train = "corpus/train.spacy"
dev   = "corpus/valid.spacy"

[system]
gpu_allocator = null

[nlp]
lang = "en"
pipeline = ["tok2vec","ner"]

[components]

[components.tok2vec]
source = "en_core_web_lg"
component = "tok2vec"

[components.ner]
factory = "ner"

[components.ner.model]
@architectures = "spacy.TransitionBasedParser.v2"
state_type = "ner"
extra_state_tokens = false
hidden_width = 64
maxout_pieces = 2
use_upper = true
nO = null

[components.ner.model.tok2vec]
@architectures = "spacy.Tok2VecListener.v1"
width = 96

[training]
dev_corpus = "corpora.dev"
train_corpus = "corpora.train"
max_epochs = 0
patience = 1600
max_steps = 3000
eval_frequency = 50
seed = 42
accumulate_gradient = 1
frozen_components = ["tok2vec"]
annotating_components = ["tok2vec"]

[training.optimizer]
@optimizers = "Adam.v1"
learn_rate = 0.001

[training.batcher]
@batchers = "spacy.batch_by_words.v1"
size = 500
tolerance = 0.2
discard_oversize = false

[corpora]

[corpora.train]
@readers = "spacy.Corpus.v1"
path = ${paths.train}
max_length = 0

[corpora.dev]
@readers = "spacy.Corpus.v1"
path = ${paths.dev}
max_length = 0

[initialize]
vectors = "en_core_web_lg"

In [ ]:
!python -m spacy init fill-config base_config.cfg config_stage1.cfg
!python -m spacy debug config config_stage1.cfg
!python -m spacy train config_stage1.cfg --output ./output_frozen

## 6. Stage-2 config (unfrozen, sources frozen model) + train

In [ ]:
%%writefile base_config_stage2.cfg
[paths]
train = "corpus/train.spacy"
dev   = "corpus/valid.spacy"

[system]
gpu_allocator = null

[nlp]
lang = "en"
pipeline = ["tok2vec","ner"]

[components]

[components.tok2vec]
source = "output_frozen/model-best"
component = "tok2vec"

[components.ner]
source = "output_frozen/model-best"
component = "ner"

[training]
dev_corpus = "corpora.dev"
train_corpus = "corpora.train"
max_epochs = 0
patience = 1600
max_steps = 8000
eval_frequency = 50
seed = 42
accumulate_gradient = 1
frozen_components = []
annotating_components = []

[training.optimizer]
@optimizers = "Adam.v1"
learn_rate = 0.0001

[training.batcher]
@batchers = "spacy.batch_by_words.v1"
size = 500
tolerance = 0.2
discard_oversize = false

[corpora]

[corpora.train]
@readers = "spacy.Corpus.v1"
path = ${paths.train}
max_length = 0

[corpora.dev]
@readers = "spacy.Corpus.v1"
path = ${paths.dev}
max_length = 0

[initialize]
vectors = "en_core_web_lg"

In [ ]:
!python -m spacy init fill-config base_config_stage2.cfg config_stage2.cfg
!python -m spacy debug config config_stage2.cfg
!python -m spacy train config_stage2.cfg --output ./output_stage2

## 7. Evaluate per-type on the held-out test set

In [ ]:
!python -m spacy evaluate ./output_stage2/model-best corpus/test.spacy --output metrics.json
import json; print(json.dumps(json.load(open("metrics.json"))["ents_per_type"], indent=2))

## 8. (Optional) smoke test
Upload `smoke_test.py`, then run it.

In [ ]:
from google.colab import files
files.upload()   # smoke_test.py
!python smoke_test.py

## 9. Download the model

In [ ]:
import shutil
from google.colab import files
shutil.make_archive("model_stage2", "zip", "output_stage2/model-best")
files.download("model_stage2.zip")